# v7 beta — class160: noise audit → noise simulation → CNN

อ่าน NIH เดิมจาก `MyDrive/nih -xray/data/3` วิเคราะห์ครบ **112,120 ภาพ** แล้วคัดภาพ estimated noise ต่ำ
Train **160+160**, validation **40+40**, test **50+50** ตาม patient partition เดิม

ทดลอง **12 วิธีเดิม ไม่มี strength L4**, noise 5 ชนิด × 3 ระดับ + ภาพก่อนเติม, seed 42
ฝึก **1 DAE + 12 CNN**; เก็บ checkpoint และโครงสำหรับ XAI แต่ **ยังไม่รัน Grad-CAM/SHAP**

1. เลือก **Runtime → Change runtime type → GPU**
2. กด **Run all** แล้วอนุญาต mount Google Drive
3. ผลและข้อมูลใช้ซ้ำอยู่ `MyDrive/nih -xray/Notebook_7_beta/`

ถ้า runtime หลุด เปิด notebook นี้แล้ว Run all ใหม่ จะใช้ audit chunks / ชุดภาพที่คัด / checkpoint บน Drive ต่อ
ไม่ต้องอัปโหลด ZIP ภาพใหม่ และ notebook มี source code ฝังไว้แล้ว

ภาพที่คัดเป็น **low estimated noise reference** ไม่อ้างว่าเป็นภาพไร้ noise จริง
รอบนี้เป็น beta pilot หนึ่ง seed ไม่ใช่ผลสรุปหลาย training seeds


In [ ]:
#@title ติดตั้งและเชื่อมต่อ Google Drive
from pathlib import Path
import base64, hashlib, subprocess, sys, zipfile, io
import importlib.util
from google.colab import drive
drive.mount("/content/drive")
dependencies = {"numpy": "numpy", "pandas": "pandas", "PIL": "Pillow", "cv2": "opencv-python-headless", "pywt": "PyWavelets", "scipy": "scipy", "sklearn": "scikit-learn", "matplotlib": "matplotlib", "nbformat": "nbformat"}
missing = [package for module, package in dependencies.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
SOURCE_SHA256 = '61d55737761d3489bddab8fb273b77d2401a05f58bf861168fc73a1843181855'
SOURCE_B64 = '...'
payload = base64.b64decode(SOURCE_B64)
if hashlib.sha256(payload).hexdigest() != SOURCE_SHA256:
    raise ValueError("Embedded source integrity failed")
SOURCE_STAGE = Path("/content/v7_source_") .with_name("v7_source_" + SOURCE_SHA256[:16])
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    for entry in archive.infolist():
        if not (SOURCE_STAGE / entry.filename).resolve().is_relative_to(SOURCE_STAGE.resolve()):
            raise ValueError("Source path escapes staging")
    archive.extractall(SOURCE_STAGE)
ROOT = SOURCE_STAGE / "Notebook_7_beta"
loaded = sys.modules.get("v7")
if loaded is not None and not Path(loaded.__file__).resolve().is_relative_to(ROOT.resolve()):
    raise RuntimeError("Notebook source changed; restart the Colab session before loading this version")
sys.path.insert(0, str(ROOT))
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU selected")


## 1. ตำแหน่งข้อมูลและผลบน Drive

In [ ]:
from v7.colab import DRIVE_DATA_ROOT, DRIVE_WORK_ROOT
import os
DATA_ROOT = Path(DRIVE_DATA_ROOT)  # MyDrive/nih -xray/data/3 (ภาพเดิม)
WORK = Path(DRIVE_WORK_ROOT)       # MyDrive/nih -xray/Notebook_7_beta (ผลและข้อมูลใช้ซ้ำ)
WORK.mkdir(parents=True, exist_ok=True)
os.environ["TORCH_HOME"] = str(WORK / "cache/torch")  # reuse ImageNet weights across runtimes
required = ["Data_Entry_2017.csv", "train_val_list.txt", "test_list.txt"]
missing = [name for name in required if not (DATA_ROOT / name).is_file()]
if missing:
    raise FileNotFoundError(f"Check DATA_ROOT={DATA_ROOT}; missing {missing}")
print("Read-only source:", DATA_ROOT)
print("Durable outputs:", WORK)


## 2. วิเคราะห์ noise และคัดภาพ — บันทึก checkpoint ลง Drive
รอบแรกอ่านภาพทั้งหมด; รอบถัดไปใช้ชุดที่คัดไว้แล้ว

In [ ]:
from v7.colab import prepare_on_drive
prepare_on_drive(ROOT, DATA_ROOT, WORK, workers=4)


## 3. โหลดชุดภาพที่คัดจาก Drive เข้า Colab VM

In [ ]:
from v7.colab import stage_selected
LOCAL_DATA = stage_selected(WORK, Path("/content/v7_cache"))
print("VM-local training images:", LOCAL_DATA)


## 4. ฝึก DAE และ CNN 12 วิธี — เก็บ checkpoint ทุก epoch บน Drive

In [ ]:
from v7.runner import run
if not torch.cuda.is_available():
    raise RuntimeError("Runtime > Change runtime type > GPU แล้ว Run all ใหม่")
EXPERIMENT = run(ROOT, LOCAL_DATA, WORK / "runs/v7_beta/seed_42", device="cuda")


## 5. ผล CNN และคุณภาพภาพ

In [ ]:
import pandas as pd
REPORTS = WORK / "runs/v7_beta/seed_42/reports"
table = pd.read_csv(REPORTS / "classification.csv")
display(table[["condition_id", "corruption_id", "roc_auc", "pr_auc", "balanced_accuracy", "f1", "accuracy"]])
print("Reports:", REPORTS)
print("Checkpoints / XAI loader metadata:", REPORTS.parent)
